# 간단한 도구(Tool)와 LLM 통합 데모

도구가 어떻게 작동하는지 아주 간단한 예제로 시작해 보겠습니다.  
여기서는 작동 방식을 직접 볼 수 있도록 노트북 안에서 도구를 만들고, 직접 호출하는 방식과 LLM을 통해 호출하는 방식으로 모두 사용해 봅니다.  
테스트할 도구는 간단한 계산기입니다. 단순하지만, LLM은 계산을 어려워하는 경우가 많기 때문에 LLM과 함께 쓰면 꽤 효과적입니다.

## 계산기 도구 만들기

도구는 본질적으로 다음과 같은 함수입니다.
- 구조화된 입력(JSON)을 받습니다
- 특정 동작을 수행합니다
- 구조화된 출력(JSON)을 반환합니다

In [ ]:
import json
from typing import Dict, Any

class CalculatorTool:
    """기본 사칙연산을 수행하는 간단한 계산기 도구입니다."""
    
    def __init__(self):
        self.name = "calculator"
        self.description = """Performs basic arithmetic operations (add, subtract, multiply, divide). 
Expected input format:
{
    "operation": "add" | "subtract" | "multiply" | "divide",
    "a": float,
    "b": float
}

Returns:
{
    "result": float
}
"""
    
    def execute(self, tool_input: Dict[str, Any]) -> Dict[str, Any]:
        """
        계산기 도구를 실행합니다.
        
        입력 형식:
        {
            "operation": "add" | "subtract" | "multiply" | "divide",
            "a": float,
            "b": float
        }
        
        반환값:
        {
            "result": float
        }
        """
        try:
            operation = tool_input.get("operation")
            a = float(tool_input.get("a", 0))
            b = float(tool_input.get("b", 0))
            
            if operation == "add":
                result = a + b
            elif operation == "subtract":
                result = a - b
            elif operation == "multiply":
                result = a * b
            elif operation == "divide":
                if b == 0:
                    return {
                        "error": "Division by zero is not allowed",
                    }
                result = a / b
            else:
                return {
                    "error": f"Unknown operation: {operation}",
                }
            
            return {
                "result": result,
            }
        except Exception as e:
            return {
                "error": str(e)
            }

# 도구 인스턴스 생성
calculator = CalculatorTool()
print(f"Tool created: {calculator.name}")
print(f"Description: {calculator.description}")

## 도구 직접 사용하기

이제 JSON 입력을 몇 개 보내고 JSON 출력을 받아 도구를 테스트해 보겠습니다.

In [ ]:
# 예제 1: 덧셈
input_json = {
    "operation": "add",
    "a": 15,
    "b": 27
}

print("Input JSON:")
print(json.dumps(input_json, indent=2))

output = calculator.execute(input_json)

print("\nOutput JSON:")
print(json.dumps(output, indent=2))

In [ ]:
# 예제 2: 곱셈
input_json = {
    "operation": "multiply",
    "a": 8,
    "b": 7
}

print("Input JSON:")
print(json.dumps(input_json, indent=2))

output = calculator.execute(input_json)

print("\nOutput JSON:")
print(json.dumps(output, indent=2))

In [ ]:
# 예제 3: 나눗셈
input_json = {
    "operation": "divide",
    "a": 100,
    "b": 4
}

print("Input JSON:")
print(json.dumps(input_json, indent=2))

output = calculator.execute(input_json)

print("\nOutput JSON:")
print(json.dumps(output, indent=2))

좋습니다! 이것이 도구가 작동하는 기본 원리입니다. 도구는 단순히 JSON 입력을 받아 JSON 출력으로 응답하는 서비스입니다.  
하지만 지금까지는 우리가 직접 호출했습니다. 그것도 좋지만 계산기는 이미 우리도 쓸 줄 알죠. 그렇다면 LLM이 이 도구를 사용하게 하려면 어떻게 해야 할까요?

## LLM을 루프에 넣기

이제 LLM을 통합하여 다음 작업을 하게 만들겠습니다.
1. 사용자의 요청을 이해합니다
2. 요청을 도구에 맞는 JSON 형식으로 만듭니다
3. 도구를 호출합니다
4. 사용자를 위해 결과를 해석하고 응답합니다

먼저 Llama Stack 클라이언트를 설정하겠습니다.

In [ ]:
from llama_stack_client import LlamaStackClient
from llama_stack_client.lib.inference.event_logger import EventLogger

# Llama Stack 서버에 연결
base_url = "http://llama-stack-service:8321"
client = LlamaStackClient(base_url=base_url)
model = "vllm-llama32/llama32"

print(f"Connected to Llama Stack at: {base_url}")
print(f"Using model: {model}")

### LLM이 사용자 요청을 해석하고 도구 호출을 생성하기

시스템 프롬프트를 작성할 때 도구 설명(description)을 활용할 수 있다는 점에 주목하세요. 나중에 도구를 점점 더 많이 추가할 때 매우 유용합니다. 프롬프트를 직접 고칠 필요 없이 도구의 설명을 참조하기만 하면 됩니다!

### 단계별 에이전트 따라가기

입력과 출력을 명확하게 볼 수 있도록 각 단계를 하나씩 따라가 보겠습니다.

**1단계: 사용자 요청 정의하기**

In [ ]:
# 사용자가 자연어로 묻는 내용
user_request = "What is 45 plus 78?"

print("USER REQUEST:")
print(user_request)

**2단계: 도구 설명을 이용해 시스템 프롬프트 만들기**

In [ ]:
# 도구 설명을 사용해 시스템 프롬프트를 동적으로 구성
system_prompt = f"""You are a helpful assistant that converts natural language requests into JSON tool calls.

Available tool: {calculator.name}
{calculator.description}

When the user asks you to perform a calculation, respond with ONLY a JSON object following the format above.
Do not include any explanation, just return the JSON object.
"""

print("SYSTEM PROMPT:")
print(system_prompt)

**3단계: LLM에 요청을 보내 도구 호출 생성하기**

In [ ]:
# 자연어 요청을 도구 호출로 변환하도록 LLM에 요청
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_request}
    ],
    stream=False,
)

# LLM의 응답 추출
llm_response = response.choices[0].message.content

print("LLM RESPONSE:")
print(llm_response)

**4단계: LLM 응답을 파싱해 도구에 전달하기**

In [ ]:
# 응답 정리(LLM이 JSON을 마크다운 코드 블록으로 감싸는 경우가 있음)
cleaned_response = llm_response.strip()
if cleaned_response.startswith("```"):
    # 마크다운 코드 블록 표시 제거
    lines = cleaned_response.split("\n")
    cleaned_response = "\n".join(lines[1:-1])
    if cleaned_response.startswith("json"):
        cleaned_response = cleaned_response[4:].strip()

# JSON 파싱
tool_call = json.loads(cleaned_response)

tool_result = calculator.execute(tool_call)

print("TOOL RESULT:")
print(json.dumps(tool_result, indent=2))

**5단계: 결과를 LLM에 다시 보내 해석하기**

In [ ]:
# 도구 결과를 사용해 사용자 질문에 답하도록 LLM에 요청
interpretation_prompt = f"""User question: {user_request}
Tool result: {json.dumps(tool_result)}

Answer the user's question directly with the result. Be concise."""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer user questions directly and concisely."},
        {"role": "user", "content": interpretation_prompt}
    ],
    stream=True,
)

print("FINAL RESPONSE TO USER:")
print("-" * 60)
for log in EventLogger().log(response):
    log.print()
print("-" * 60)

응답을 받았습니다. 좋습니다!  
늘 그렇듯이 다른 어조의 응답을 원한다면 프롬프트를 자유롭게 바꿔 보세요.

### 전체 도구 워크플로 함수

각 단계를 하나씩 살펴보았으니, 이제 모든 과정을 한 번에 수행하는 재사용 가능한 함수로 묶어 보겠습니다!  
이 함수의 이름을 `run_agent`라고 지은 이유는, 지금 만든 것이 모델을 여러 번(2번) 자동으로 호출하고 도구를 사용하는 아주 간단한 에이전트 워크플로이기 때문입니다.  
압니다.. 아직은 그다지 자율적이지 않죠. 하지만 조금만 기다려 주세요, 곧 그렇게 될 거예요 ;)

In [ ]:
def run_agent(user_request: str):
    """
    전체 에이전트 흐름:
    1. 사용자가 자연어로 요청함
    2. LLM이 요청을 해석해 도구 호출 형식으로 만듦
    3. 도구가 실행됨
    4. LLM이 결과를 해석해 사용자에게 응답함
    """
    print("="*60)
    print(f"USER REQUEST: {user_request}")
    print("="*60)

    # 1단계: 도구 설명을 사용해 시스템 프롬프트 구성
    system_prompt = f"""You are a helpful assistant that converts natural language requests into JSON tool calls.

Available tool: {calculator.name}
{calculator.description}

When the user asks you to perform a calculation, respond with ONLY a JSON object following the format above.
Do not include any explanation, just return the JSON object.
"""

    # 2단계: LLM이 도구 호출 형식을 만듦
    print("\n[Step 1] LLM interpreting request...")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_request}
        ],
        stream=False,
    )

    llm_response = response.choices[0].message.content

    # 3단계: 응답 파싱
    try:
        cleaned_response = llm_response.strip()
        if cleaned_response.startswith("```"):
            lines = cleaned_response.split("\n")
            cleaned_response = "\n".join(lines[1:-1])
            if cleaned_response.startswith("json"):
                cleaned_response = cleaned_response[4:].strip()

        tool_call = json.loads(cleaned_response)
    except json.JSONDecodeError as e:
        print(f"Error parsing LLM response: {e}")
        print(f"LLM response was: {llm_response}")
        return

    print("\n[Step 2] Generated Tool Call:")
    print(json.dumps(tool_call, indent=2))

    # 3단계: 도구 실행
    print("\n[Step 3] Executing tool... Tool Results:")
    tool_result = calculator.execute(tool_call)
    print(json.dumps(tool_result, indent=2))

    # 4단계: LLM이 결과를 해석
    print("\n[Step 4] LLM interpreting results for user...")
    interpretation_prompt = f"""User question: {user_request}
Tool result: {json.dumps(tool_result)}

Answer the user's question directly with the result. Be concise."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer user questions directly and concisely."},
            {"role": "user", "content": interpretation_prompt}
        ],
        stream=True,
    )
    print("-" * 60)
    for log in EventLogger().log(response):
        log.print()
    print("-" * 60)
    print()

### 실행해 봅시다! 🎉

이제 자연어로 요청하면 LLM이 다음을 수행합니다.
1. 우리가 원하는 것을 이해합니다
2. 적절한 도구 호출 형식으로 만듭니다
3. 도구를 실행합니다
4. 결과를 설명합니다

In [ ]:
# 예제 1: 간단한 덧셈
run_agent("What is 45 plus 78?")

In [ ]:
# 예제 2: 곱셈
run_agent("Can you multiply 12 by 15 for me?")

오류 처리도 테스트해 보겠습니다. 예를 들어 0으로 나누면 어떻게 될까요? 🔥

In [ ]:
# 예제 3: 나눗셈
run_agent("I need to divide 144 by 0")

In [ ]:
# 예제 4: 뺄셈
run_agent("What's 100 minus 37?")

## 요약

이 노트북에서 배운 내용은 다음과 같습니다.

1. **도구 만들기**: 도구는 다음과 같은 간단한 클래스/함수입니다.
   - 특정 파라미터가 담긴 JSON 입력을 받습니다
   - 동작(여기서는 계산)을 수행합니다
   - 결과가 담긴 JSON 출력을 반환합니다

2. **도구 직접 사용하기**: 올바른 형식의 JSON을 제공하면 도구를 직접 호출할 수 있습니다

3. **LLM을 루프에 넣기**: LLM은 다음 역할을 하는 지능형 인터페이스입니다.
   - 사용자 요청을 해석합니다
   - 요청을 올바른 도구 호출 형식으로 만듭니다
   - 사용자를 위해 도구 결과를 해석합니다
   - 텍스트로 응답합니다

이 패턴은 LLM이 사용자 의도를 이해하고, 적절한 도구를 호출하고, 의미 있는 응답을 제공하는 시스템인 **AI 에이전트**의 기초입니다! 🚀